In [ ]:
import gc
from pathlib import Path

from shapely.geometry import box

import cyanomembranes as cm

(OUT := Path("output")).mkdir(exist_ok=True)
import sys
from pathlib import Path

# Add the folder that CONTAINS membrane_analysis/ to the path
sys.path.insert(0, str(Path("..").resolve()))  # adjust if needed


In [ ]:
from Bio.PDB import PDBParser, NeighborSearch
import numpy as np


In [ ]:
def get_pockets(file_path, target_chains, ligand_name, cutoff=5.0):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", file_path)
    model = structure[0]

    # --- collect every matching ligand residue ---
    ligands = []
    for chain in model:
        if target_chains is None or chain.id in target_chains:
            for residue in chain:
                if residue.resname == ligand_name:
                    ligands.append(residue)

    print(f"Found {len(ligands)} {ligand_name} instance(s) in chains {target_chains}")

    # --- neighbour search over the whole structure ---
    all_atoms = list(structure.get_atoms())
    ns = NeighborSearch(all_atoms)

    # --- one pocket entry per ligand instance ---
    all_pockets = {}
    for lig in ligands:
        chain_id  = lig.get_parent().id
        res_seq   = lig.id[1]          # sequence number, e.g. 501
        key       = (chain_id, res_seq)

        pocket_residues = set()
        for atom in lig.get_atoms():
            for nearby_atom in ns.search(atom.coord, cutoff):
                neighbour = nearby_atom.get_parent()
                if neighbour is not lig and neighbour.id[0] == " ":
                    pocket_residues.add(neighbour)

        all_pockets[(chain_id, res_seq)] = pocket_residues
        print(f"  Chain {chain_id}, {ligand_name} {res_seq}: "
              f"{len(pocket_residues)} pocket residues")

    return all_pockets


# --- usage ---
pockets = get_pockets("../output/protein_shadows/4H13-cytb6f_wo.pdb",
                      target_chains=None,
                      ligand_name="PL9")

# iterate results
for (chain, seqnum), residues in pockets.items():
    print(f"\nPocket for PL9 {seqnum} in chain {chain}:")
    for r in sorted(residues, key=lambda r: r.id[1]):
        print(f"  {r.get_parent().id} {r.resname} {r.id[1]}")

In [ ]:
def get_atom_coords(chain_res_lst):
    atom_coords = []

    for c in [chain_res_lst]:
        chain = pockets[c]
        for res in chain:
            for atom in res.get_atoms():
                atom_coords.append(atom.coord[:2])

    atom_coords = np.array(atom_coords)
    return atom_coords

atom_coords = get_atom_coords(("I", 306))

In [ ]:
PDB_FILES = [
    "6RQF-Cyt-spinach.pdb"
]

DATA = Path("../tutorial_plants/data_plants")
(OUT := Path("../tutorial_plants/outputs")).mkdir(exist_ok=True)
(PIC_DIR :=  (OUT/"pictures_membranes")).mkdir(exist_ok=True)

proteins = cm.pdb_utils.process_proteins(DATA, OUT / "protein_shadows", PDB_FILES)


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
psii = proteins["6RQF-Cyt-spinach"]["polygon"][0]
ax.plot(*psii.exterior.xy)
ax.set_aspect("equal")
ax.scatter(atom_coords[:,0], atom_coords[:,1], s=1)


In [ ]:
from shapely.geometry import Point
from shapely.ops import nearest_points

def project_on_nperimeter(polygon, coords):
    cx, cy = psii.centroid.x, psii.centroid.y
    centroid = np.array([cx, cy])
    projected = []
    for pt in atom_coords:
        p = Point(pt)
        nearest = nearest_points(psii.boundary, p)[0]
        projected.append((nearest.x, nearest.y))
        
    projected = np.array(projected)
    cx, cy = psii.centroid.x, psii.centroid.y
    angles = np.arctan2(projected[:, 1] - cy,
                            projected[:, 0] - cx)  # in [-π, π]

    mean_angle = np.arctan2(np.sin(angles).mean(),
                        np.cos(angles).mean())
        
    angle_diff = np.abs(np.arctan2(np.sin(angles - mean_angle),
                                       np.cos(angles - mean_angle)))

    THRESHOLD = np.deg2rad(45)  # adjust if needed
    mask = angle_diff < THRESHOLD
    return projected[mask]

projected_filtered = project_on_nperimeter(psii, atom_coords)



In [ ]:
center = projected_filtered.mean(axis=0)

# Vector from polygon centroid to the cluster center
cx, cy = psii.centroid.x, psii.centroid.y
direction = center - np.array([cx, cy])
direction = direction / np.linalg.norm(direction)   # normalize

# Snap the touch point onto the perimeter
touch_point = nearest_points(psii.boundary, Point(center))[0]
touch_pt_arr = np.array([touch_point.x, touch_point.y])

# Move the circle center outward by radius
circle_center = touch_pt_arr + direction * radius

shapely_circle = Point(circle_center).buffer(radius)

fig, ax = plt.subplots()
psii = proteins["6RQF-Cyt-spinach"]["polygon"][0]
ax.plot(*psii.exterior.xy)
ax.set_aspect("equal")
ax.scatter(atom_coords[:, 0], atom_coords[:, 1], s=1)
ax.scatter(projected_filtered[:, 0], projected_filtered[:, 1], s=5)
ax.plot(*shapely_circle.exterior.xy)


In [ ]:
PDB_FILES = [
    "1JB0-PSI-syn-cocc.pdb",
    "1NEK-SDH-Ecoli.pdb",
    "1OCO-cytoxidase-bov.trpdb",
    "1xl4-Kchannel-Pmagnetotacticum.trpdb",
    "3WU2-PSII-ThermosynVul.pdb",
    "4H13-cytb6f.trpdb",
]

DATA = Path("../data_cyano/")
(OUT := Path("../output")).mkdir(exist_ok=True)
(PIC_DIR :=  (OUT/"pictures_membranes")).mkdir(exist_ok=True)

proteins = cm.pdb_utils.process_proteins(DATA, OUT / "protein_shadows", PDB_FILES)

In [ ]:
from shapely import Point
import matplotlib.pyplot as plt 

cytb6f = proteins['4H13-cytb6f']["polygon"][0]
p1 = Point(-40, 0).buffer(5)
p2 = Point(40, 0).buffer(5)

fig,ax = plt.subplots()
ax.plot(*cytb6f.exterior.xy)
ax.plot(*p1.exterior.xy)
ax.plot(*p2.exterior.xy)
ax.set_aspect("equal")
plt.show()

In [ ]:
import numpy as np
from shapely.geometry import Point
from shapely import affinity



def define_anchor_point(polygon, p1, p2):
    boundary = polygon.exterior
    d1 = boundary.project(Point(p1), normalized=True)
    d2 = boundary.project(Point(p2), normalized=True)
    anchor1 = np.array(boundary.interpolate(d1, normalized=True).coords[0])
    anchor2 = np.array(boundary.interpolate(d2, normalized=True).coords[0])
    arm1 = p1 - anchor1
    arm2 = p2 - anchor2
    return {"d1": d1, "d2": d2, "arm1": arm1, "arm2": arm2,
               "anchor1_orig": anchor1, "anchor2_orig": anchor2}


anchor_info = define_anchor_point(cytb6f, np.array([-40, 0]), np.array([40, 0]))


def place_points(polygon, anchor_info, radius=5):
    d1, d2 = anchor_info["d1"], anchor_info["d2"]
    arm1, arm2 = anchor_info["arm1"], anchor_info["arm2"]
    anchor1_orig = anchor_info["anchor1_orig"]
    anchor2_orig = anchor_info["anchor2_orig"]

    boundary_rot = cytb6f_rotated.exterior
    anchor1_rot = np.array(boundary_rot.interpolate(d1, normalized=True).coords[0])
    anchor2_rot = np.array(boundary_rot.interpolate(d2, normalized=True).coords[0])


    def rotate_arm(anchor_orig, anchor_rot, arm):
        v_orig = anchor_orig - np.array(polygon.centroid.coords[0])
        v_rot  = anchor_rot  - np.array(polygon.centroid.coords[0])
        angle = np.arctan2(v_rot[1], v_rot[0]) - np.arctan2(v_orig[1], v_orig[0])
        R = np.array([[np.cos(angle), -np.sin(angle)],
                      [np.sin(angle),  np.cos(angle)]])
        return anchor_rot + R @ arm

    p1 = Point(*rotate_arm(anchor1_orig, anchor1_rot, arm1)).buffer(radius)
    p2 = Point(*rotate_arm(anchor2_orig, anchor2_rot, arm2)).buffer(radius)
    return p1, p2

# ── STEP 3: use it for many rotated instances ─────────────────────────────────
angles = [0, 30, 47, 90, 135]  # only used to CREATE test cases, not passed to place_points

fig, axes = plt.subplots(1, len(angles), figsize=(4 * len(angles), 4))

for ax, angle_deg in zip(axes, angles):
    cytb6f_rotated = affinity.rotate(cytb6f, angle_deg, origin='centroid')
    p1, p2 = place_points(cytb6f_rotated, anchor_info)   # no angle_deg !

    ax.plot(*cytb6f_rotated.exterior.xy, 'b-')
    ax.plot(*p1.exterior.xy, 'r-')
    ax.plot(*p2.exterior.xy, 'g-')
    ax.set_title(f'{angle_deg}°')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def get_complex_idx(p):
    psi_area = proteins["1JB0-PSI-syn-cocc"]["polygon"][0].area
    psii_area = proteins["3WU2-PSII-ThermosynVul"]["polygon"][0].area
    cytb6f_area = proteins["4H13-cytb6f"]["polygon"][0].area
    psii_idx = []
    psi_idx = []
    psimono_idx =  []
    cytb6f_idx = []
    for idx, i in enumerate(p):
        if np.isclose(i.area, psi_area):
            psi_idx.append(idx)
        if np.isclose(i.area, psii_area):
            psii_idx.append(idx)
        if np.isclose(i.area, cytb6f_area):
            cytb6f_idx.append(idx)
    return {"PSI": psi_idx, "PSII": psii_idx, "Cytb6f": cytb6f_idx}


p = cm.geo_utils.readwkt("../output/avg_membrane/polygons_avg_membrane_110-0.wkt")

idxes = get_complex_idx(p)
binding_sites = []

for idx in idxes["Cytb6f"][:10]:
    fig, ax = plt.subplots()
    cytb6f_rotated = p[idx]
    p1, p2 = place_points(cytb6f_rotated, anchor_info)
    binding_sites.extend([p1, p2])
    ax.plot(*cytb6f_rotated.exterior.xy, 'b-')
    ax.plot(*p1.exterior.xy, 'r-')
    ax.plot(*p2.exterior.xy, 'g-')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    plt.show()



In [ ]:
from shapely.ops import unary_union
unary_union(binding_sites)